# Lecture 3: Types of Cross-Validation

### Short, simple, self-study notes

Cross-validation helps us estimate how well a model will work on unseen data and helps us choose hyperparameters such as alpha, k, or model settings.

**Main rule:** Keep the final test set untouched. Use cross-validation only inside the training data.

## 1. The train, validation, and test idea

A useful workflow is:

1. Split the complete dataset into a training set and a final test set.
2. Use only the training set for learning and hyperparameter tuning.
3. Cross-validation repeatedly creates training and validation parts inside that training set.
4. After choosing the model, evaluate it once on the untouched test set.

The validation score helps us compare models. The test score is the final honest estimate of performance on new data.

**Do not use the test set while tuning.** If the test set influences decisions, it is no longer a fair final test.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle

fig, ax = plt.subplots(figsize=(12, 2.8))
ax.set_xlim(0, 12)
ax.set_ylim(0, 3)
ax.axis("off")

def block(x, width, label, color):
    ax.add_patch(Rectangle((x, 1.0), width, 1.0, facecolor=color, edgecolor="black"))
    ax.text(x + width / 2, 1.5, label, ha="center", va="center", fontsize=11, weight="bold")

block(0.2, 2.2, "Complete\ndata", "#d9eaf7")
ax.add_patch(FancyArrowPatch((2.5, 1.5), (3.4, 1.5), arrowstyle="->", mutation_scale=15))
block(3.5, 5.0, "Training data\nused for cross-validation", "#d9f2d9")
block(8.8, 2.7, "Final test data\nused once", "#f9d5d3")
ax.add_patch(FancyArrowPatch((8.5, 1.5), (8.75, 1.5), arrowstyle="->", mutation_scale=15))
ax.text(6.0, 0.35, "CV repeatedly splits this green block into train and validation folds", ha="center", fontsize=10)
ax.set_title("Keep the final test set separate from model selection", fontsize=13, weight="bold")
plt.show()


## 2. Why not use just one validation split?

With one random train-validation split, the score can change when the random state changes. One split may give 85% accuracy, another 92%, and another 78% simply because different rows were selected.

Cross-validation uses several splits. We train and validate several times, then look at the average score and its variation.

**Better summary:** mean validation score ± standard deviation.

This gives a more stable estimate than trusting one lucky or unlucky split.

## 3. K-Fold Cross-Validation

In K-Fold CV:

1. Split the training data into K parts called folds.
2. Train on K-1 folds and validate on the remaining fold.
3. Repeat until every fold has been the validation fold once.
4. Average the K validation scores.

Example: with 500 training rows and K=5, each validation fold has about 100 rows. The model trains five times, and every row is used for validation once.

**Common choice:** K=5 or K=10. A larger K uses more data for training but needs more model fits.

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold

n_records = 20
X_demo = np.arange(n_records).reshape(-1, 1)
y_demo = np.array([0] * 12 + [1] * 8)  # A binary classification label.

def fold_grid(splitter, X, y=None):
    """Return 0=train, 1=validation, and -1=not used for each fold."""
    grid = np.full((splitter.get_n_splits(X, y), len(X)), -1)
    for row, (train_index, validation_index) in enumerate(splitter.split(X, y)):
        grid[row, train_index] = 0
        grid[row, validation_index] = 1
    return grid

plain_kfold = KFold(n_splits=5, shuffle=True, random_state=7)
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

fig, axes = plt.subplots(2, 1, figsize=(12, 4.5), sharex=True)
for ax, grid, title in zip(
    axes,
    [fold_grid(plain_kfold, X_demo), fold_grid(stratified_kfold, X_demo, y_demo)],
    ["K-Fold: rows are shuffled into folds", "Stratified K-Fold: class proportions are preserved"],
):
    ax.imshow(grid, aspect="auto", cmap="Pastel1", vmin=-1, vmax=1)
    ax.set_yticks(range(5), ["Fold 1", "Fold 2", "Fold 3", "Fold 4", "Fold 5"])
    ax.set_ylabel("validation round")
    ax.set_title(title)
axes[-1].set_xlabel("record number")
fig.suptitle("Each row shows one train-validation split (orange = validation)", y=1.02)
plt.tight_layout()
plt.show()


### How to read the K-Fold diagram

- Each row is one experiment.
- The validation block changes from row to row.
- All rows together allow every record to be validated once.
- Shuffling is useful when row order has no meaning. Use a fixed random state for repeatable results.

## 4. Leave-One-Out Cross-Validation (LOOCV)

Leave-One-Out means the validation fold contains exactly one row.

For N rows, the model is trained N times:

- Round 1: validate on row 1, train on all other rows.
- Round 2: validate on row 2, train on all other rows.
- Continue until every row has been held out once.

**Advantage:** Almost all available data is used for training in every round.

**Disadvantages:** It can be very slow for large datasets, and the validation scores can be noisy. It is usually not the first choice for large datasets.

## 5. Leave-P-Out Cross-Validation

Leave-P-Out is a general version of Leave-One-Out. Instead of leaving one row out, we leave P rows out for validation.

Example: with 10 rows and P=2, each validation set contains 2 rows. The process tries many possible groups of 2 rows.

The number of possible splits is the combination:

**number of splits = C(N, P)**

This can grow very quickly, so Leave-P-Out is expensive. K-Fold is usually more practical.

## 6. Stratified K-Fold Cross-Validation

Stratified K-Fold is designed mainly for **classification**. It tries to keep the same class proportions in every fold.

Example: if the complete training data has 60% class 0 and 40% class 1, each validation fold will be close to 60% class 0 and 40% class 1.

Why is this useful? Without stratification, one fold might accidentally contain mostly one class. That gives an unfair or unstable validation score, especially when classes are imbalanced.

- Use **KFold** when row order is safe to shuffle and class balance is not the issue, often for regression.
- Use **StratifiedKFold** for classification when class proportions matter.
- Never stratify a time series by randomly mixing past and future rows.

## 7. Time-Series Cross-Validation

Time-series data has an order: yesterday comes before today, and today comes before tomorrow. We must not randomly mix future observations into the training set.

Time-series CV uses earlier observations for training and later observations for validation:

- Round 1: train on early history, validate on the next time period.
- Round 2: train on more history, validate on the following period.
- Continue moving forward in time.

This better matches the real task: use the past to predict the future. In scikit-learn, TimeSeriesSplit provides this type of split.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

time_values = np.arange(12).reshape(-1, 1)
time_splitter = TimeSeriesSplit(n_splits=4)
time_grid = np.full((time_splitter.get_n_splits(time_values), len(time_values)), -1)

for row, (train_index, validation_index) in enumerate(time_splitter.split(time_values)):
    time_grid[row, train_index] = 0
    time_grid[row, validation_index] = 1

plt.figure(figsize=(12, 3))
plt.imshow(time_grid, aspect="auto", cmap="Pastel1", vmin=-1, vmax=1)
plt.yticks(range(4), ["Round 1", "Round 2", "Round 3", "Round 4"])
plt.xticks(range(12), [f"t{i + 1}" for i in range(12)])
plt.xlabel("time moves from past to future")
plt.ylabel("validation round")
plt.title("TimeSeriesSplit: training uses the past, validation uses the future")
plt.show()


### How to read the time-series diagram

- The validation section is always after the training section.
- The training window expands as more history becomes available.
- There is no random shuffling.
- Use this for sales, stock measurements, sensor readings, website traffic, and other time-ordered data.

## 8. A small scikit-learn example

The code below uses K-Fold CV to evaluate a Ridge model. The model is fitted separately inside every fold. The scaler is inside the pipeline, so each fold learns scaling only from its own training rows.

In [ ]:
from sklearn.datasets import make_regression, make_classification
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X_reg, y_reg = make_regression(n_samples=120, n_features=5, noise=12, random_state=7)
ridge_model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
regression_cv = KFold(n_splits=5, shuffle=True, random_state=7)
r2_scores = cross_val_score(ridge_model, X_reg, y_reg, cv=regression_cv, scoring="r2")

X_clf, y_clf = make_classification(
    n_samples=120, n_features=5, weights=[0.75, 0.25], random_state=7
)
logistic_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2_000))
classification_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
accuracy_scores = cross_val_score(logistic_model, X_clf, y_clf, cv=classification_cv, scoring="accuracy")

print("Ridge R2 scores:", np.round(r2_scores, 3))
print("Ridge mean R2:", round(r2_scores.mean(), 3), "+/-", round(r2_scores.std(), 3))
print("Logistic regression accuracy scores:", np.round(accuracy_scores, 3))
print("Logistic mean accuracy:", round(accuracy_scores.mean(), 3), "+/-", round(accuracy_scores.std(), 3))


### What does cross_val_score return?

- One score for each validation fold.
- The mean is the average performance across folds.
- The standard deviation shows how much the score changes between folds.
- The scoring metric must match the problem: accuracy for classification, R² or negative error metrics for regression.

For model selection, tools such as GridSearchCV or RandomizedSearchCV can test many hyperparameter values using cross-validation. After selecting the best settings, use the untouched test set once.

## 9. Quick comparison table

| Method | Main idea | Good use | Main warning |
|---|---|---|---|
| K-Fold | K rotating validation folds | General regression/classification | Shuffle only when row order has no meaning |
| Leave-One-Out | One validation row per round | Very small datasets | N model fits can be slow and noisy |
| Leave-P-Out | P validation rows per round | Special small-data experiments | C(N, P) splits can explode |
| Stratified K-Fold | Preserves class proportions | Classification, especially imbalanced classes | Needs class labels |
| TimeSeriesSplit | Past trains, future validates | Time-ordered data | Never randomly shuffle time data |

### One-line memory trick

**K-Fold rotates groups, LOOCV leaves one out, LPO leaves P out, Stratified protects class ratios, and TimeSeriesSplit protects time order.**

## 10. Final revision card

- Cross-validation estimates generalization and helps tune hyperparameters.
- Split off the test set first; do not use it during tuning.
- K-Fold trains K times and validates on a different fold each time.
- LOOCV uses one row for validation at a time and can be expensive.
- Leave-P-Out is even more expensive because it tries many combinations.
- Stratified K-Fold keeps class proportions similar in every fold.
- Time-series CV uses past data for training and future data for validation.
- Put preprocessing inside a pipeline to avoid data leakage.
- Report the mean score and the spread across folds.

### Interview answer

**We use cross-validation to get a more reliable validation estimate and tune hyperparameters, while keeping the final test data untouched for the final performance check.**